# Lab 66: The Loss Aversion

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 7, *The Decision Bias*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-66.ipynb)

**What you will do:** find your own loss-aversion threshold and mug-experiment prices, compare them with the book's figures, pool the class's data, and run a toy simulation of a risk-neutral versus a loss-averse agent facing the same repeated bet.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 66 you offered a friend a bet: heads they win £12, tails they lose £10 — a positive
expected value of +£1, which a purely rational agent would always accept. Most people decline,
and the book reports a typical acceptance threshold of about £20 for a £10 loss, a ratio of
roughly **2:1** — **loss aversion** (Kahneman & Tversky, 1979). In the mug variation, sellers
who were given a mug typically demanded more to give it up than buyers were willing to pay for
the same mug — the **endowment effect** (Kahneman, Knetsch, & Thaler, 1990). The book is careful
to flag that Gal and Rucker (2018) have challenged how universal the fixed "2:1" figure really
is: effect sizes vary a great deal across studies, and some careful studies find gains and losses
weighted almost equally.

## Record your results

Enter the smallest win *G* your friend would need to accept a 50-50 bet against a £10 loss, and
the mug-experiment prices from a seller and a buyer.

In [ ]:
my_G = 22               # example data: the smallest win (in £) your friend would accept against a £10 loss
seller_price = 8.0       # example data: what your "seller" wanted for the mug
buyer_price = 4.5        # example data: what your "buyer" offered for the same mug

print(f"Your bet threshold: accept if win >= £{my_G} against a £10 loss")
print(f"Mug experiment: seller asked £{seller_price}, buyer offered £{buyer_price}")

## Compare

The book states a typical threshold of about **£20** for a £10 loss (a **2:1** ratio). For the
mug experiment, it gives no specific prices, only that sellers typically demand more than
buyers will pay — so that half of the comparison is about direction, not magnitude.

In [ ]:
my_ratio = my_G / 10
print(f"Your ratio: {my_ratio:.1f}:1   |   book's typical ratio: ~2:1 (about £20 for a £10 loss)")
if my_ratio > 1:
    print("Your friend needed more than an even-money win to accept the bet, consistent with")
    print("loss aversion -- though Gal and Rucker (2018) note the exact ratio varies a great")
    print("deal across people and contexts, so do not expect it to land exactly on 2.0.")
else:
    print("Your friend accepted at (or below) even odds -- this would be an unusually")
    print("loss-*tolerant* result by the book's account, though Gal and Rucker's critique")
    print("means such variation is not as surprising as the classic '2:1' figure suggests.")
print()
if seller_price > buyer_price:
    print(f"Seller (£{seller_price}) asked more than buyer (£{buyer_price}) offered -- the")
    print("same direction as the endowment effect the book describes.")
else:
    print(f"Seller (£{seller_price}) did not ask more than buyer (£{buyer_price}) offered --")
    print("a reversal of the usual pattern, which a small, informal trial can certainly produce.")

## The AI side

The book asks whether an AI agent should be built to be loss-averse, noting this is "both an
opportunity ... and a risk," depending on context. Below, a risk-neutral agent that always
accepts the £12/£10 bet is compared with a loss-averse agent that uses the same prospect-theory
rule as Lab 65's simulation (alpha = 0.88, lambda = 2.25) to decide whether to play, across many
repeated rounds of the identical bet.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
rng = np.random.default_rng(2026)
win, lose, alpha, lamb, n_rounds = 12, 10, 0.88, 2.25, 300

def prospect_theory_accepts(win, lose, alpha, lamb):
    return 0.5 * (win ** alpha) > 0.5 * lamb * (lose ** alpha)

risk_neutral_accepts = True  # a risk-neutral agent always plays because the expected value is positive
loss_averse_accepts = prospect_theory_accepts(win, lose, alpha, lamb)
print(f"Risk-neutral agent plays every round: {risk_neutral_accepts}")
print(f"Loss-averse agent (prospect-theory rule) plays every round: {loss_averse_accepts}")

outcomes = rng.choice([win, -lose], size=n_rounds, p=[0.5, 0.5])
risk_neutral_money = np.cumsum(outcomes) if risk_neutral_accepts else np.zeros(n_rounds)
loss_averse_money = np.cumsum(outcomes) if loss_averse_accepts else np.zeros(n_rounds)

plt.figure(figsize=(6, 3.5))
plt.plot(risk_neutral_money, color="steelblue", label="risk-neutral agent")
plt.plot(loss_averse_money, color="firebrick", label="loss-averse agent")
plt.xlabel("round"); plt.ylabel("cumulative money (£)"); plt.legend(); plt.show()
print(f"After {n_rounds} rounds: risk-neutral agent has £{risk_neutral_money[-1]:.0f}, "
      f"loss-averse agent has £{loss_averse_money[-1]:.0f}")

Because this particular bet has a positive expected value and no catastrophic downside, the
loss-averse agent's caution costs it money over many rounds -- exactly the "counterproductive"
case the book raises for domains like financial trading. The same loss-averse rule would look
like a safety feature, not a flaw, in a domain where a single bad outcome (a crash, an
accident) is unacceptable regardless of the average payoff. Whether loss aversion helps or
hurts an AI agent is a property of the task and the stakes, not of the mathematics alone.

## Pool the class data

Pool the class's bet thresholds *G* (anonymous IDs, not names) to see how much the estimated
loss-aversion ratio varies from person to person.

In [ ]:
import io
csv_text = """id,G
P01,18
P02,25
P03,20
P04,15
P05,30
P06,20
P07,12
"""  # example data: replace with your class CSV, e.g. pd.read_csv("loss_aversion_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
df["ratio"] = df.G / 10
display(df)
boot = [df.ratio.sample(frac=1, replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean ratio: {df.ratio.mean():.2f}:1 (95% bootstrap interval {lo:.2f}-{hi:.2f}, n = {len(df)})")
print("Book's typical value: ~2:1. Gal and Rucker (2018) report this ratio varies a great deal")
print("across studies, with some finding gains and losses weighted almost equally.")
df.ratio.plot.hist(bins=8, color="firebrick", alpha=0.7, figsize=(6, 3))
plt.axvline(2.0, color="black", linestyle="--", label="book's typical ratio (2:1)")
plt.xlabel("individual ratio (G / 10)"); plt.legend(); plt.show()

## Questions to think about

1. Was your friend's threshold *G* close to the book's typical £20 figure, well above it, or well below it? What might explain a large difference?
2. The book reports that Gal and Rucker (2018) found "a few carefully designed studies find gains and losses weighted almost equally." What kind of study design do you think would be needed to tell a genuine 1:1 result apart from a poorly measured 2:1 result?
3. In the endowment-effect mug variation, why might merely holding an object for a few minutes change how much you would demand to give it up?
4. In the risk-neutral versus loss-averse simulation, what happens to the gap between the two agents' final totals if you increase the number of rounds? What does that tell you about the long-run cost of excessive caution when the odds are genuinely in your favour?

## Challenge

Vary the stakes systematically with one friend: offer the £10-loss bet with increasing possible
wins (£11, £13, £16, £20, £25) until they accept, to find their threshold more precisely than a
single guess.

In [ ]:
# Example data: replace with your own staircase of offers and accept/decline responses.
offers = [11, 13, 16, 20, 25]
accepted = [False, False, False, True, True]   # example data: whether your friend accepted each offer

staircase = pd.DataFrame({"offer": offers, "accepted": accepted})
display(staircase)
threshold = staircase.loc[staircase.accepted, "offer"].min()
print(f"Estimated threshold: the smallest accepted win was £{threshold}, a ratio of {threshold / 10:.1f}:1")
print("A staircase like this pins down the threshold more precisely than a single yes/no guess.")
print("The offers and responses above are example data -- replace them with your own.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-66.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")